# IFRS S1/S2 Requirement Extraction with GPT-5.2

This notebook extracts IFRS S1/S2 disclosure requirements into structured JSON objects that can be used by your report-generation agents.

It is designed to be run **cell by cell** so you can debug each step:
1. Install/import packages
2. Configure model, paths, and PDFs
3. Parse PDFs
4. Create chunks
5. Preview chunks before sending them to the model
6. Test extraction on one chunk
7. Run full extraction with checkpointing
8. Deduplicate, validate, and export
9. Generate a reusable `requirements_kb.py`
10. Test retrieval by report section
11. Plug the requirements into a section agent prompt

> Important: this notebook assumes you have local IFRS PDFs in your project folder. Put them in `data/standards/`.


## 0. Folder structure expected

Create this structure before running the full extraction:

```text
project/
│
├── data/
│   ├── standards/
│   │   ├── ifrs_s1.pdf
│   │   ├── ifrs_s2.pdf
│   │   ├── ifrs_s2_accompanying_guidance.pdf
│   │   └── illustrative_examples.pdf
│   │
│   └── requirements/
│
└── notebooks/
    └── 01_extract_ifrs_requirements_gpt52_cell_by_cell.ipynb
```

You can rename the PDF files, but if you do, update the `INPUT_PDFS` cell.


In [ ]:
# 1. Install packages
# Run this once in your notebook environment.

%pip install -q --upgrade openai pydantic pypdf tenacity pandas


In [ ]:
# 2. Imports

import os
import re
import json
import time
from pathlib import Path
from collections import defaultdict, Counter
from typing import Literal

import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError
from pypdf import PdfReader
from tenacity import retry, stop_after_attempt, wait_exponential
from openai import OpenAI

print("Imports loaded successfully.")


In [ ]:
# 3. Configuration

# If GPT-5.2 is available in your account, keep this.
# If not, replace it with a model available to you, for example: "gpt-5.5", "gpt-5.4", or "gpt-5.4-mini".
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.2")

# Use smaller chunks while debugging. Increase later if needed.
MAX_CHARS_PER_CHUNK = 12_000

# Set this to a small number while testing. Set to None for all chunks.
MAX_CHUNKS_PER_DOCUMENT = None

# Output folders
BASE_DIR = Path(".")
STANDARDS_DIR = BASE_DIR / "data" / "standards"
OUTPUT_DIR = BASE_DIR / "data" / "requirements"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = OUTPUT_DIR / "ifrs_requirements.json"
OUTPUT_RAW_JSONL = OUTPUT_DIR / "ifrs_requirements_raw_checkpoint.jsonl"
OUTPUT_KB = OUTPUT_DIR / "requirements_kb.py"

# Your input PDFs
INPUT_PDFS = [
    {
        "path": STANDARDS_DIR / "ifrs_s1.pdf",
        "standard": "S1",
        "source_doc": "IFRS S1 General Requirements",
    },
    {
        "path": STANDARDS_DIR / "ifrs_s2.pdf",
        "standard": "S2",
        "source_doc": "IFRS S2 Climate-related Disclosures",
    },
    {
        "path": STANDARDS_DIR / "ifrs_s2_accompanying_guidance.pdf",
        "standard": "S2_AG",
        "source_doc": "IFRS S2 Accompanying Guidance",
    },
    {
        "path": STANDARDS_DIR / "illustrative_examples.pdf",
        "standard": "IE",
        "source_doc": "IFRS S1/S2 Illustrative Examples",
    },
]

print("Model:", MODEL)
print("Standards folder:", STANDARDS_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())


In [ ]:
# 4. Check that your PDFs exist

missing = []

for item in INPUT_PDFS:
    path = item["path"]
    if path.exists():
        print(f"FOUND: {path}")
    else:
        print(f"MISSING: {path}")
        missing.append(path)

if missing:
    print("\nSome PDFs are missing. Add them to data/standards/ or update INPUT_PDFS.")
else:
    print("\nAll PDFs found.")


In [ ]:
# 5. OpenAI client setup

# Recommended: set your key in your environment:
# Windows CMD:
#   set OPENAI_API_KEY=your_key_here
#
# PowerShell:
#   $env:OPENAI_API_KEY="your_key_here"
#
# Linux/Mac:
#   export OPENAI_API_KEY=your_key_here

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is not set yet.")
    print("Set it in your environment before running API cells.")
else:
    print("OPENAI_API_KEY found.")

client = OpenAI()


In [ ]:
# 6. Structured output schema

class IFRSRequirement(BaseModel):
    model_config = ConfigDict(extra="forbid")

    standard: Literal["S1", "S2", "S2_AG", "IE"]
    source_doc: str

    paragraph: str
    section: Literal[
        "general_requirements",
        "governance",
        "strategy",
        "risk_management",
        "metrics_targets",
        "industry_metrics",
        "other",
    ]

    obligation_type: Literal["shall", "should", "may"]

    requirement_text: str
    applies_to_banks: bool
    related_paragraphs: list[str]
    metric_type: str | None

    page_start: int
    page_end: int


class RequirementExtractionResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    requirements: list[IFRSRequirement]


print("Pydantic schemas ready.")


In [ ]:
# 7. PDF parsing helpers

def clean_text(text: str) -> str:
    """Basic PDF text cleanup."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    """Extract text page by page from a PDF."""
    reader = PdfReader(str(pdf_path))
    pages = []

    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = clean_text(text)

        if text:
            pages.append({
                "page": i,
                "text": text,
            })

    return pages


def make_chunks(pages: list[dict], max_chars: int = MAX_CHARS_PER_CHUNK) -> list[dict]:
    """Group PDF pages into chunks while preserving page range metadata."""
    chunks = []
    current_text = []
    current_pages = []
    current_len = 0

    for page in pages:
        page_text = f"\n\n[PAGE {page['page']}]\n{page['text']}"

        if current_text and current_len + len(page_text) > max_chars:
            chunks.append({
                "text": "\n".join(current_text),
                "page_start": min(current_pages),
                "page_end": max(current_pages),
                "char_count": current_len,
            })

            current_text = []
            current_pages = []
            current_len = 0

        current_text.append(page_text)
        current_pages.append(page["page"])
        current_len += len(page_text)

    if current_text:
        chunks.append({
            "text": "\n".join(current_text),
            "page_start": min(current_pages),
            "page_end": max(current_pages),
            "char_count": current_len,
        })

    return chunks


print("PDF parsing helpers ready.")


In [ ]:
# 8. Test PDF parsing on the first available PDF

available_pdfs = [x for x in INPUT_PDFS if x["path"].exists()]

if not available_pdfs:
    raise FileNotFoundError("No PDFs found. Add PDFs to data/standards/ first.")

test_pdf = available_pdfs[0]
pages = extract_pdf_pages(test_pdf["path"])
chunks = make_chunks(pages)

print("Test document:", test_pdf["source_doc"])
print("Pages extracted:", len(pages))
print("Chunks created:", len(chunks))

if pages:
    print("\nFirst page preview:")
    print(pages[0]["text"][:1500])


In [ ]:
# 9. Preview chunks before sending to GPT

chunk_preview_rows = []

for pdf_info in available_pdfs:
    pages = extract_pdf_pages(pdf_info["path"])
    chunks = make_chunks(pages)

    for idx, chunk in enumerate(chunks, start=1):
        chunk_preview_rows.append({
            "source_doc": pdf_info["source_doc"],
            "standard": pdf_info["standard"],
            "chunk_index": idx,
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "char_count": chunk["char_count"],
            "text_preview": chunk["text"][:250].replace("\n", " "),
        })

chunk_df = pd.DataFrame(chunk_preview_rows)
chunk_df.head(10)


In [ ]:
# 10. Prompts

SYSTEM_PROMPT = """
You are an IFRS S1/S2 disclosure requirement extraction specialist.

Your job is to extract disclosure requirements from IFRS sustainability reporting text.

Extract only actual disclosure obligations or application guidance that affects disclosure.
A requirement is usually indicated by wording such as:
- shall disclose
- shall include
- shall describe
- shall provide
- shall explain
- is required to disclose
- should disclose
- may disclose

Do not invent requirements.
Do not paraphrase the paragraph text.
The requirement_text must be copied from the provided text as closely as possible.

Classify each item into one section:
- general_requirements
- governance
- strategy
- risk_management
- metrics_targets
- industry_metrics
- other

Use industry_metrics for SASB / industry-based banking metrics.

Set applies_to_banks to true when the text refers to:
- banks
- commercial banks
- financial institutions
- lending
- financed emissions
- credit exposure
- mortgages
- loans
- investment portfolios
- any other banking or finance-specific disclosure concept

Return only the structured output.
""".strip()


def build_user_prompt(
    text_chunk: str,
    standard: str,
    source_doc: str,
    page_start: int,
    page_end: int,
) -> str:
    return f"""
STANDARD: {standard}
SOURCE DOCUMENT: {source_doc}
PAGE RANGE: {page_start}-{page_end}

Extract every relevant disclosure requirement from the text below.

For every extracted item:
- standard must be "{standard}"
- source_doc must be "{source_doc}"
- page_start must be {page_start}
- page_end must be {page_end}
- paragraph must be the paragraph number if visible, for example "29", "B63", "C1", "FN-CB-410a.1", or "unknown"
- obligation_type must be "shall", "should", or "may"
- related_paragraphs must be an empty list if no cross-reference is present
- metric_type must be null if no specific metric is required

TEXT:
{text_chunk}
""".strip()


print("Prompts ready.")


In [ ]:
# 11. GPT extraction function for one chunk

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=20))
def extract_requirements_from_chunk(
    text_chunk: str,
    standard: str,
    source_doc: str,
    page_start: int,
    page_end: int,
) -> list[dict]:
    """Extract requirements from one text chunk using OpenAI Structured Outputs."""

    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": build_user_prompt(
                    text_chunk=text_chunk,
                    standard=standard,
                    source_doc=source_doc,
                    page_start=page_start,
                    page_end=page_end,
                ),
            },
        ],
        text_format=RequirementExtractionResult,
    )

    parsed = response.output_parsed

    if parsed is None:
        raise ValueError("The model response could not be parsed into RequirementExtractionResult.")

    return [req.model_dump() for req in parsed.requirements]


print("Extraction function ready.")


In [ ]:
# 12. DEBUG: run extraction on only one chunk first

# This cell calls the OpenAI API.
# It is the best cell to debug model access, schema problems, and output quality.

pdf_info = available_pdfs[0]
pages = extract_pdf_pages(pdf_info["path"])
chunks = make_chunks(pages)

test_chunk_index = 0
test_chunk = chunks[test_chunk_index]

print("Testing on:")
print("Document:", pdf_info["source_doc"])
print("Chunk:", test_chunk_index + 1)
print("Pages:", test_chunk["page_start"], "-", test_chunk["page_end"])
print("Characters:", test_chunk["char_count"])

test_requirements = extract_requirements_from_chunk(
    text_chunk=test_chunk["text"],
    standard=pdf_info["standard"],
    source_doc=pdf_info["source_doc"],
    page_start=test_chunk["page_start"],
    page_end=test_chunk["page_end"],
)

print(f"Requirements found: {len(test_requirements)}")

pd.DataFrame(test_requirements).head(20)


In [ ]:
# 13. Inspect one extracted requirement in full

if not test_requirements:
    print("No requirements found in the test chunk.")
else:
    i = 0
    print(json.dumps(test_requirements[i], indent=2, ensure_ascii=False))


In [ ]:
# 14. Checkpoint helpers

def chunk_key(source_doc: str, chunk_index: int, page_start: int, page_end: int) -> str:
    return f"{source_doc}::chunk={chunk_index}::pages={page_start}-{page_end}"


def load_processed_chunk_keys(checkpoint_path: Path) -> set[str]:
    """Load chunk keys that were already processed."""
    if not checkpoint_path.exists():
        return set()

    keys = set()
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            keys.add(record["chunk_key"])

    return keys


def append_checkpoint_records(checkpoint_path: Path, key: str, requirements: list[dict]) -> None:
    """Append extracted requirements for one chunk to a JSONL checkpoint file."""
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        for req in requirements:
            f.write(json.dumps({
                "chunk_key": key,
                "requirement": req,
            }, ensure_ascii=False) + "\n")


def load_checkpoint_requirements(checkpoint_path: Path) -> list[dict]:
    """Load all requirements from the JSONL checkpoint file."""
    if not checkpoint_path.exists():
        return []

    requirements = []
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            requirements.append(record["requirement"])

    return requirements


print("Checkpoint helpers ready.")


In [ ]:
# 15. FULL RUN: extract all requirements with checkpointing

# This cell calls the OpenAI API many times.
# While debugging, set MAX_CHUNKS_PER_DOCUMENT = 1 or 2 in the config cell.
# When you are confident, set MAX_CHUNKS_PER_DOCUMENT = None and rerun.

processed_keys = load_processed_chunk_keys(OUTPUT_RAW_JSONL)
print(f"Already processed chunks in checkpoint: {len(processed_keys)}")

total_new_chunks = 0

for pdf_info in available_pdfs:
    pdf_path = pdf_info["path"]
    standard = pdf_info["standard"]
    source_doc = pdf_info["source_doc"]

    print("\n" + "=" * 80)
    print(f"Processing: {source_doc}")
    print(f"File: {pdf_path}")

    pages = extract_pdf_pages(pdf_path)
    chunks = make_chunks(pages)

    if MAX_CHUNKS_PER_DOCUMENT is not None:
        chunks = chunks[:MAX_CHUNKS_PER_DOCUMENT]

    print(f"Pages extracted: {len(pages)}")
    print(f"Chunks to process: {len(chunks)}")

    for idx, chunk in enumerate(chunks, start=1):
        key = chunk_key(source_doc, idx, chunk["page_start"], chunk["page_end"])

        if key in processed_keys:
            print(f"Skipping already processed chunk {idx}/{len(chunks)} pages {chunk['page_start']}-{chunk['page_end']}")
            continue

        print(f"Extracting chunk {idx}/{len(chunks)} pages {chunk['page_start']}-{chunk['page_end']}")

        try:
            reqs = extract_requirements_from_chunk(
                text_chunk=chunk["text"],
                standard=standard,
                source_doc=source_doc,
                page_start=chunk["page_start"],
                page_end=chunk["page_end"],
            )

            append_checkpoint_records(OUTPUT_RAW_JSONL, key, reqs)
            processed_keys.add(key)
            total_new_chunks += 1

            print(f"  Found {len(reqs)} requirements")

            # Gentle pause to reduce rate-limit pressure.
            time.sleep(0.5)

        except Exception as e:
            print(f"  ERROR on chunk {idx}: {type(e).__name__}: {e}")
            print("  You can fix the issue and rerun this cell; checkpointed chunks will be skipped.")
            raise

print("\nFull run complete.")
print("New chunks processed:", total_new_chunks)
print("Checkpoint file:", OUTPUT_RAW_JSONL)


In [ ]:
# 16. Load raw checkpoint results

raw_requirements = load_checkpoint_requirements(OUTPUT_RAW_JSONL)

print("Raw extracted requirements:", len(raw_requirements))

if raw_requirements:
    raw_df = pd.DataFrame(raw_requirements)
    display(raw_df.head(20))
else:
    print("No checkpoint requirements found yet.")


In [ ]:
# 17. Deduplication and sorting

def normalize_text_for_dedupe(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def deduplicate_requirements(requirements: list[dict]) -> list[dict]:
    seen = set()
    deduped = []

    for req in requirements:
        key = (
            req.get("standard"),
            req.get("paragraph"),
            normalize_text_for_dedupe(req.get("requirement_text", ""))[:350],
        )

        if key not in seen:
            seen.add(key)
            deduped.append(req)

    return deduped


def sort_requirements(requirements: list[dict]) -> list[dict]:
    section_order = {
        "general_requirements": 0,
        "governance": 1,
        "strategy": 2,
        "risk_management": 3,
        "metrics_targets": 4,
        "industry_metrics": 5,
        "other": 6,
    }

    obligation_order = {
        "shall": 0,
        "should": 1,
        "may": 2,
    }

    return sorted(
        requirements,
        key=lambda r: (
            r.get("standard", ""),
            section_order.get(r.get("section", "other"), 99),
            obligation_order.get(r.get("obligation_type", "may"), 99),
            r.get("page_start", 999999),
            str(r.get("paragraph", "")),
        ),
    )


deduped_requirements = deduplicate_requirements(raw_requirements)
final_requirements = sort_requirements(deduped_requirements)

print("Raw requirements:", len(raw_requirements))
print("After deduplication:", len(final_requirements))


In [ ]:
# 18. Validate final requirements against the schema

valid_requirements = []
invalid_requirements = []

for req in final_requirements:
    try:
        valid_req = IFRSRequirement.model_validate(req)
        valid_requirements.append(valid_req.model_dump())
    except ValidationError as e:
        invalid_requirements.append({
            "requirement": req,
            "errors": e.errors(),
        })

print("Valid requirements:", len(valid_requirements))
print("Invalid requirements:", len(invalid_requirements))

if invalid_requirements:
    print("\nExample invalid requirement:")
    print(json.dumps(invalid_requirements[0], indent=2, ensure_ascii=False))


In [ ]:
# 19. Save final JSON

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(valid_requirements, f, ensure_ascii=False, indent=2)

print(f"Saved final requirements JSON to: {OUTPUT_JSON.resolve()}")
print(f"Total saved requirements: {len(valid_requirements)}")


In [ ]:
# 20. Quality-control summaries

if not valid_requirements:
    print("No valid requirements to summarize yet.")
else:
    df = pd.DataFrame(valid_requirements)

    print("By section:")
    display(df["section"].value_counts().rename_axis("section").reset_index(name="count"))

    print("By standard:")
    display(df["standard"].value_counts().rename_axis("standard").reset_index(name="count"))

    print("By obligation type:")
    display(df["obligation_type"].value_counts().rename_axis("obligation_type").reset_index(name="count"))

    print("Bank-specific requirements:")
    display(df["applies_to_banks"].value_counts().rename_axis("applies_to_banks").reset_index(name="count"))

    display(df.head(20))


In [ ]:
# 21. Inspect requirements by section

SECTION_TO_INSPECT = "governance"

section_reqs = [
    r for r in valid_requirements
    if r["section"] == SECTION_TO_INSPECT
]

print(f"Section: {SECTION_TO_INSPECT}")
print(f"Requirements: {len(section_reqs)}")

for r in section_reqs[:10]:
    print("\n" + "-" * 80)
    print(f"{r['standard']} ¶{r['paragraph']} | {r['obligation_type']} | bank={r['applies_to_banks']}")
    print(r["requirement_text"][:1000])


In [ ]:
# 22. Generate requirements_kb.py

def write_requirements_kb(requirements: list[dict], output_path: Path) -> None:
    grouped = defaultdict(list)

    for req in requirements:
        grouped[req["section"]].append(req)

    grouped = dict(grouped)

    kb_code = f'''"""
Auto-generated IFRS S1/S2 requirements knowledge base.
Do not edit manually. Regenerate from the extraction notebook.
"""

REQUIREMENTS = {repr(grouped)}


def get_requirements(
    section: str,
    standard: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
) -> list[dict]:
    """Return requirements by section, optionally filtered by standard/bank relevance."""

    reqs = REQUIREMENTS.get(section, [])

    if standard:
        reqs = [r for r in reqs if r["standard"] == standard]

    if mandatory_only:
        reqs = [r for r in reqs if r["obligation_type"] == "shall"]

    if banks_only:
        reqs = [r for r in reqs if r["applies_to_banks"]]

    return sorted(
        reqs,
        key=lambda r: (
            r["obligation_type"] != "shall",
            r["standard"],
            r["page_start"],
            r["paragraph"],
        )
    )


def list_sections() -> list[str]:
    return sorted(REQUIREMENTS.keys())


def count_requirements() -> int:
    return sum(len(v) for v in REQUIREMENTS.values())
'''

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(kb_code)


write_requirements_kb(valid_requirements, OUTPUT_KB)

print(f"Saved Python KB to: {OUTPUT_KB.resolve()}")


In [ ]:
# 23. Test the generated KB without importing it

# This simulates how your report agents will query the KB.

requirements_by_section = defaultdict(list)

for req in valid_requirements:
    requirements_by_section[req["section"]].append(req)


def get_requirements_from_memory(
    section: str,
    standard: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
) -> list[dict]:
    reqs = list(requirements_by_section.get(section, []))

    if standard:
        reqs = [r for r in reqs if r["standard"] == standard]

    if mandatory_only:
        reqs = [r for r in reqs if r["obligation_type"] == "shall"]

    if banks_only:
        reqs = [r for r in reqs if r["applies_to_banks"]]

    return sorted(
        reqs,
        key=lambda r: (
            r["obligation_type"] != "shall",
            r["standard"],
            r["page_start"],
            r["paragraph"],
        )
    )


test_governance_reqs = get_requirements_from_memory(
    section="governance",
    mandatory_only=True,
)

print("Mandatory governance requirements:", len(test_governance_reqs))

for r in test_governance_reqs[:5]:
    print(f"- [{r['standard']} ¶{r['paragraph']}] {r['requirement_text'][:250]}...")


In [ ]:
# 24. Build a section-agent prompt using the extracted requirements

def build_section_prompt(section_name: str, payload: dict, requirements: list[dict]) -> str:
    mandatory_reqs = [
        r for r in requirements
        if r["section"] == section_name and r["obligation_type"] == "shall"
    ]

    req_text = "\n".join(
        f"[{r['standard']} ¶{r['paragraph']}] {r['requirement_text']}"
        for r in mandatory_reqs
    )

    return f"""
You are writing the {section_name.replace("_", " ").title()} section of an IFRS S1/S2 sustainability report for a bank.

MANDATORY DISCLOSURE REQUIREMENTS:
You must address every requirement below.

{req_text}

BANK DATA:
{json.dumps(payload, indent=2, ensure_ascii=False)}

Instructions:
1. Write a professional sustainability disclosure section.
2. Use only the bank data provided.
3. Do not invent metrics.
4. If data is insufficient for a requirement, explicitly state the missing data.
5. Reference IFRS paragraph numbers where relevant.
6. Keep the tone close to a real annual sustainability report.
""".strip()


# Example fake payload just to test prompt construction.
sample_governance_payload = {
    "bank_name": "Example Bank",
    "reporting_year": 2024,
    "board_oversight": {
        "board_climate_meetings": 6,
        "climate_reports_frequency": "semi-annual",
        "responsible_committee": "Risk Committee",
    },
    "management_role": {
        "executive_owner": "Chief Risk Officer",
        "climate_risk_team": "Enterprise Risk Management",
    },
}

governance_prompt = build_section_prompt(
    section_name="governance",
    payload=sample_governance_payload,
    requirements=valid_requirements,
)

print(governance_prompt[:4000])


In [ ]:
# 25. Optional: generate one report section with GPT

# This cell calls the OpenAI API.
# Run it only after you are satisfied with the prompt preview above.

def generate_section_draft(section_name: str, payload: dict, requirements: list[dict]) -> str:
    prompt = build_section_prompt(
        section_name=section_name,
        payload=payload,
        requirements=requirements,
    )

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": "You are an IFRS S1/S2 sustainability reporting specialist for banking.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    return response.output_text


# Uncomment to test:
# governance_draft = generate_section_draft(
#     section_name="governance",
#     payload=sample_governance_payload,
#     requirements=valid_requirements,
# )
#
# print(governance_draft)


In [ ]:
# 26. Build a simple coverage checklist for QA

def build_coverage_checklist(section_name: str, requirements: list[dict]) -> pd.DataFrame:
    section_reqs = [
        r for r in requirements
        if r["section"] == section_name and r["obligation_type"] == "shall"
    ]

    rows = []

    for r in section_reqs:
        rows.append({
            "standard": r["standard"],
            "paragraph": r["paragraph"],
            "section": r["section"],
            "applies_to_banks": r["applies_to_banks"],
            "metric_type": r["metric_type"],
            "requirement_text": r["requirement_text"],
            "covered_in_draft": None,
            "evidence_sentence": None,
            "missing_data": None,
        })

    return pd.DataFrame(rows)


governance_checklist = build_coverage_checklist("governance", valid_requirements)
governance_checklist.head(20)


In [ ]:
# 27. Save checklist template

CHECKLIST_PATH = OUTPUT_DIR / "governance_coverage_checklist_template.csv"

if len(governance_checklist) > 0:
    governance_checklist.to_csv(CHECKLIST_PATH, index=False, encoding="utf-8-sig")
    print(f"Saved checklist template to: {CHECKLIST_PATH.resolve()}")
else:
    print("No governance checklist rows to save yet.")


## How to use this notebook in your pipeline

After you run the full extraction, use these two files in your project:

```text
data/requirements/ifrs_requirements.json
data/requirements/requirements_kb.py
```

Your section agents should not rely only on similarity search. They should receive:
1. The bank payload for the section
2. The mandatory IFRS/SASB requirements for the same section
3. Instructions to either satisfy each requirement or state missing data

This makes the system easier to defend in your PFE because each generated paragraph can be traced back to disclosure obligations.
